# K-Means Clustering: Step-by-Step Demo

This notebook illustrates how the K-Means algorithm works by:
1. Generating synthetic 2D data with known cluster structure
2. Implementing K-Means **from scratch** so you can see each step
3. Visualizing every iteration (centroid movement and cluster assignments)
4. Comparing our implementation with scikit-learn's `KMeans`
5. Using the **Elbow Method** to choose the right number of clusters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from IPython.display import display, clear_output
import time

## 1. Generate Synthetic Data

We create 300 points arranged in 4 clusters so the algorithm has clear structure to discover.

In [ ]:
np.random.seed(42)

X, y_true = make_blobs(
    n_samples=300,
    centers=4,
    cluster_std=0.9,
    random_state=42,
)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=30, alpha=0.7, edgecolors='k', linewidths=0.5)
plt.title('Raw Data (No Labels)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

## 2. K-Means from Scratch

The algorithm repeats two steps until convergence:

| Step | Name | What happens |
|------|------|--------------|
| 1 | **Assign** | Each point is assigned to the nearest centroid |
| 2 | **Update** | Each centroid moves to the mean of its assigned points |

Below is a clean implementation that records the history of centroids and labels at every iteration so we can visualize the process.

In [ ]:
def kmeans_scratch(X, k, max_iters=20, seed=0):
    """Run K-Means and return the history of centroids and labels."""
    rng = np.random.RandomState(seed)

    # --- Initialise centroids by picking k random data points ---
    indices = rng.choice(len(X), size=k, replace=False)
    centroids = X[indices].copy()

    history = []  # store (centroids, labels) at each step

    for i in range(max_iters):
        # Step 1: Assign each point to nearest centroid
        #   distances shape: (n_samples, k)
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)

        history.append((centroids.copy(), labels.copy()))

        # Step 2: Update centroids
        new_centroids = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
            for j in range(k)
        ])

        # Check for convergence
        if np.allclose(centroids, new_centroids):
            print(f'Converged after {i + 1} iterations.')
            break

        centroids = new_centroids
    else:
        print(f'Stopped after {max_iters} iterations (did not converge).')

    return centroids, labels, history

In [ ]:
K = 4
final_centroids, final_labels, history = kmeans_scratch(X, k=K, seed=10)

## 3. Visualize Every Iteration

Each subplot shows one iteration. The large **X** markers are the centroids; points are colored by their current cluster assignment. Watch how the centroids migrate toward the true cluster centers.

In [ ]:
colors = ['#e6194b', '#3cb44b', '#4363d8', '#f58231']
cmap = ListedColormap(colors[:K])

n_iters = len(history)
cols = min(n_iters, 4)
rows = (n_iters + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))
if n_iters == 1:
    axes = np.array([axes])
axes = np.atleast_2d(axes)

for idx, (centroids_i, labels_i) in enumerate(history):
    ax = axes[idx // cols, idx % cols]
    ax.scatter(X[:, 0], X[:, 1], c=labels_i, cmap=cmap,
               s=25, alpha=0.6, edgecolors='k', linewidths=0.3)
    ax.scatter(centroids_i[:, 0], centroids_i[:, 1],
               c=colors[:K], s=250, marker='X', edgecolors='black', linewidths=1.5)
    ax.set_title(f'Iteration {idx + 1}', fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])

# Hide unused subplots
for idx in range(n_iters, rows * cols):
    axes[idx // cols, idx % cols].axis('off')

fig.suptitle('K-Means Iteration-by-Iteration', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 4. Animated Version (runs in Colab)

This cell replays the iterations as an in-place animation so you can watch the centroids move in real time.

In [ ]:
for idx, (centroids_i, labels_i) in enumerate(history):
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(X[:, 0], X[:, 1], c=labels_i, cmap=cmap,
               s=30, alpha=0.6, edgecolors='k', linewidths=0.3)
    ax.scatter(centroids_i[:, 0], centroids_i[:, 1],
               c=colors[:K], s=300, marker='X', edgecolors='black', linewidths=2)
    ax.set_title(f'K-Means — Iteration {idx + 1} / {n_iters}', fontsize=14)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    plt.tight_layout()
    plt.show()
    time.sleep(1.0)

## 5. Final Result

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Our scratch implementation
axes[0].scatter(X[:, 0], X[:, 1], c=final_labels, cmap=cmap,
                s=30, alpha=0.7, edgecolors='k', linewidths=0.3)
axes[0].scatter(final_centroids[:, 0], final_centroids[:, 1],
                c=colors[:K], s=250, marker='X', edgecolors='black', linewidths=1.5)
axes[0].set_title('Our Implementation (from scratch)', fontsize=13)

# scikit-learn
km = KMeans(n_clusters=K, random_state=42, n_init=10)
sk_labels = km.fit_predict(X)
axes[1].scatter(X[:, 0], X[:, 1], c=sk_labels, cmap=cmap,
                s=30, alpha=0.7, edgecolors='k', linewidths=0.3)
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                c=colors[:K], s=250, marker='X', edgecolors='black', linewidths=1.5)
axes[1].set_title('scikit-learn KMeans', fontsize=13)

for ax in axes:
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('Comparison: Scratch vs scikit-learn', fontsize=15)
plt.tight_layout()
plt.show()

## 6. Elbow Method — Choosing *k*

We run K-Means for k = 1 to 10 and plot the **inertia** (sum of squared distances from each point to its centroid). The "elbow" in the curve suggests a good choice of *k*.

In [ ]:
inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'o-', linewidth=2, markersize=8)
plt.axvline(x=4, color='red', linestyle='--', label='k = 4 (true number)')
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Inertia (within-cluster sum of squares)', fontsize=12)
plt.title('Elbow Method', fontsize=14)
plt.xticks(K_range)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Apply K-Means to the Iris Dataset

Let's try K-Means on a real dataset — the classic Iris data (also available in this repo as `iris.data`).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

km_iris = KMeans(n_clusters=3, random_state=42, n_init=10)
iris_labels = km_iris.fit_predict(X_iris_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means clusters (using first two features for visualization)
scatter1 = axes[0].scatter(X_iris[:, 0], X_iris[:, 1], c=iris_labels,
                           cmap='viridis', s=40, alpha=0.7, edgecolors='k', linewidths=0.3)
axes[0].set_title('K-Means Clusters (k=3)', fontsize=13)
axes[0].set_xlabel(iris.feature_names[0])
axes[0].set_ylabel(iris.feature_names[1])

# True labels
scatter2 = axes[1].scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris,
                           cmap='viridis', s=40, alpha=0.7, edgecolors='k', linewidths=0.3)
axes[1].set_title('True Species Labels', fontsize=13)
axes[1].set_xlabel(iris.feature_names[0])
axes[1].set_ylabel(iris.feature_names[1])

plt.suptitle('K-Means on Iris Dataset', fontsize=15)
plt.tight_layout()
plt.show()

## Key Takeaways

| Topic | Detail |
|-------|--------|
| **Algorithm** | Alternates between assigning points to nearest centroid and recomputing centroids |
| **Convergence** | Guaranteed to converge, but may find a local minimum |
| **Initialization** | Results depend on initial centroids — run multiple times (`n_init`) |
| **Choosing k** | Use the Elbow Method, Silhouette Score, or domain knowledge |
| **Scaling** | Always standardize features when they have different units/ranges |
| **Limitations** | Assumes spherical clusters of similar size; struggles with non-convex shapes |